# 36 — Grand Ensemble v6

Extends v5 with protein-aware and auxiliary-task models from nb26–nb35.
Uses ElasticNetCV (L1+L2) meta-learner with nested scaffold CV — same
honest evaluation protocol as v2–v5.

New candidate models:
- nb26: Single-conc pseudo-labels augmented LGBM
- nb27: NR-weighted LGBM (phylogenetic transfer)
- nb28: Auxiliary features LGBM (PXR ligand sim + NN pEC50 + CRC measurements)
- nb30: Morgan + ESM-2 multi-NR LGBM
- nb31: ChemBERTa + ESM-2 multi-NR LGBM
- nb32: Morgan + ProtBERT multi-NR LGBM
- nb33: Cross-attention ChemBERTa tokens × ESM-2 residues
- nb34: Cross-attention GROVER-large × ESM-2 residues
- nb35: Chemprop 6-head auxiliary (Emax, pEC50_null, logP, TPSA, PXR-sim)

Dynamic loading: models are included only if their OOF .npy files exist.

In [1]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler

from pxr.data import load_train, load_test
from pxr.chem import bemis_murcko
from pxr.eval import scaffold_kfold_indices, rae as rae_fn
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

SEED = 42; N_FOLDS = 5
train = load_train()
te    = load_test()
y_tr  = train['pec50'].values

scaffolds = train['smiles'].map(bemis_murcko).tolist()
splits    = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)
print('Setup complete.')

Setup complete.


In [2]:
# ── 2. Model registry — (oof_path, test_path, label) ─────────────────────────
ALL_MODELS = [
    # ── Grand v5 base models ──────────────────────────────────────────────────
    ('oof_lgbm_base.npy',              'te_lgbm_base.npy',              'lgbm_base'),
    ('oof_lgbm_tuned.npy',             'te_lgbm_tuned.npy',             'lgbm_tuned'),
    ('oof_knn.npy',                    'te_knn.npy',                    'knn'),
    ('oof_chemberta.npy',              'te_chemberta.npy',              'chemberta_mlm'),
    ('oof_chemberta_mtr.npy',          'te_chemberta_mtr.npy',          'chemberta_mtr'),
    ('oof_bert_smiles.npy',            'te_bert_smiles.npy',            'bert_smiles'),
    ('oof_unimol.npy',                 'te_unimol.npy',                 'unimol'),
    ('oof_grover.npy',                 'te_grover.npy',                 'grover_base'),
    ('oof_grover_large.npy',           'te_grover_large.npy',           'grover_large'),
    # ── New models (nb26–nb35) ─────────────────────────────────────────────────
    ('oof_singleconc.npy',             'te_singleconc.npy',             'singleconc_lgbm'),
    ('oof_nr_weighted.npy',            'te_nr_weighted.npy',            'nr_weighted_lgbm'),
    ('oof_aux_features.npy',           'te_aux_features.npy',           'aux_features_lgbm'),
    ('oof_morgan_esm2_nr.npy',         'te_morgan_esm2_nr.npy',         'morgan_esm2_nr'),
    ('oof_chemberta_esm2_nr.npy',      'te_chemberta_esm2_nr.npy',      'chemberta_esm2_nr'),
    ('oof_morgan_protbert_nr.npy',     'te_morgan_protbert_nr.npy',     'morgan_protbert_nr'),
    ('oof_crossattn_chemberta_esm2.npy','te_crossattn_chemberta_esm2.npy','crossattn_chem_esm2'),
    ('oof_crossattn_grover_esm2.npy',  'te_crossattn_grover_esm2.npy', 'crossattn_grover_esm2'),
    ('oof_chemprop_aux.npy',           'te_chemprop_aux.npy',           'chemprop_aux'),
]

# Load only models whose OOF + test files both exist
loaded = []
for oof_f, te_f, label in ALL_MODELS:
    op = DATA_PROCESSED / oof_f
    tp = DATA_PROCESSED / te_f
    if op.exists() and tp.exists():
        oof_arr = np.load(str(op))
        te_arr  = np.load(str(tp))
        loaded.append((oof_arr, te_arr, label))
        print(f'  ✓ {label:35s}  OOF RAE={rae_fn(y_tr, oof_arr):.4f}')
    else:
        print(f'  ✗ {label:35s}  (missing — skipped)')

print(f'\n{len(loaded)} models loaded for ensemble.')

  ✓ lgbm_base                            OOF RAE=0.5600
  ✓ lgbm_tuned                           OOF RAE=0.5394
  ✓ knn                                  OOF RAE=0.7341
  ✓ chemberta_mlm                        OOF RAE=0.6782
  ✓ chemberta_mtr                        OOF RAE=0.5993
  ✓ bert_smiles                          OOF RAE=0.7150
  ✓ unimol                               OOF RAE=0.7008
  ✓ grover_base                          OOF RAE=0.6355
  ✓ grover_large                         OOF RAE=0.6295
  ✓ singleconc_lgbm                      OOF RAE=0.6003
  ✓ nr_weighted_lgbm                     OOF RAE=0.5964
  ✓ aux_features_lgbm                    OOF RAE=0.2179
  ✓ morgan_esm2_nr                       OOF RAE=0.5763
  ✓ chemberta_esm2_nr                    OOF RAE=0.6186
  ✓ morgan_protbert_nr                   OOF RAE=0.5749
  ✗ crossattn_chem_esm2                  (missing — skipped)
  ✓ crossattn_grover_esm2                OOF RAE=0.6139
  ✓ chemprop_aux                         OO

In [3]:
# ── 3. Build feature matrices from OOF arrays ─────────────────────────────────
if len(loaded) < 2:
    raise RuntimeError('Need at least 2 models. Run nb26–nb35 first.')

oof_matrix = np.column_stack([m[0] for m in loaded])  # (N_tr, n_models)
te_matrix  = np.column_stack([m[1] for m in loaded])  # (N_te, n_models)
labels     = [m[2] for m in loaded]

print(f'OOF matrix: {oof_matrix.shape}  Test matrix: {te_matrix.shape}')

OOF matrix: (4139, 17)  Test matrix: (513, 17)


In [4]:
# ── 4. Nested scaffold CV — honest meta-learner evaluation ────────────────────
# Inner: ElasticNetCV on (train fold) oof predictions → fit weights
# Outer: apply weights to val fold OOF predictions → get honest OOF

ALPHAS  = np.logspace(-4, 1, 30)
L1_RATIOS = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0]

oof_meta = np.full(len(y_tr), np.nan)

for fold_i, (tr_idx, va_idx) in enumerate(splits):
    X_inner = oof_matrix[tr_idx]  # inner train: model OOF predictions
    y_inner = y_tr[tr_idx]
    X_val   = oof_matrix[va_idx]

    scaler = StandardScaler()
    X_inner_s = scaler.fit_transform(X_inner)
    X_val_s   = scaler.transform(X_val)

    # Inner 5-fold CV for alpha/l1_ratio selection
    en = ElasticNetCV(alphas=ALPHAS, l1_ratio=L1_RATIOS, cv=5,
                      fit_intercept=True, max_iter=5000, random_state=SEED)
    en.fit(X_inner_s, y_inner)
    oof_meta[va_idx] = en.predict(X_val_s)

nested_rae = rae_fn(y_tr, oof_meta)
print(f'\nNested CV meta-learner RAE: {nested_rae:.4f}')
print(f'Grand v5 nested RAE:        0.5356')
print(f'Delta:                       {nested_rae - 0.5356:+.4f}')

np.save(DATA_PROCESSED / 'oof_grand_v6.npy', oof_meta)


Nested CV meta-learner RAE: 0.2189
Grand v5 nested RAE:        0.5356
Delta:                       -0.3167


In [5]:
# ── 5. Full-data ElasticNetCV for final weights ────────────────────────────────
scaler_full = StandardScaler()
X_full_s    = scaler_full.fit_transform(oof_matrix)
X_te_s      = scaler_full.transform(te_matrix)

en_full = ElasticNetCV(alphas=ALPHAS, l1_ratio=L1_RATIOS, cv=5,
                       fit_intercept=True, max_iter=5000, random_state=SEED)
en_full.fit(X_full_s, y_tr)

# Report model weights
coefs = en_full.coef_
total_abs = np.abs(coefs).sum()
weight_pct = np.abs(coefs) / (total_abs + 1e-12) * 100
print(f'\nFull-data ElasticNetCV  alpha={en_full.alpha_:.5f}  l1={en_full.l1_ratio_:.2f}')
print('\nFinal model weights:')
order = np.argsort(weight_pct)[::-1]
for i in order:
    sign = '+' if coefs[i] > 0 else '-'
    print(f'  {labels[i]:35s}  {sign}{weight_pct[i]:.1f}%')

# Test predictions
te_preds = en_full.predict(X_te_s)
te_preds = np.clip(te_preds, y_tr.min()-0.5, y_tr.max()+0.5)
np.save(DATA_PROCESSED / 'te_grand_v6.npy', te_preds)


Full-data ElasticNetCV  alpha=0.00530  l1=1.00

Final model weights:
  aux_features_lgbm                    +97.6%
  lgbm_tuned                           +1.1%
  chemprop_aux                         +1.0%
  chemberta_mlm                        +0.2%
  grover_large                         +0.0%
  morgan_esm2_nr                       -0.0%
  chemberta_esm2_nr                    -0.0%
  crossattn_grover_esm2                -0.0%
  morgan_protbert_nr                   -0.0%
  singleconc_lgbm                      -0.0%
  nr_weighted_lgbm                     -0.0%
  grover_base                          -0.0%
  unimol                               -0.0%
  chemberta_mtr                        -0.0%
  bert_smiles                          -0.0%
  knn                                  -0.0%
  lgbm_base                            -0.0%


In [6]:
# ── 6. Blend with Chemprop-08 (inverse-RAE weighting) ────────────────────────
# Load chemprop OOF + test if available
chemprop_oof_path = DATA_PROCESSED / 'oof_chemprop.npy'
chemprop_te_path  = DATA_PROCESSED / 'te_chemprop.npy'

if chemprop_oof_path.exists() and chemprop_te_path.exists():
    chemprop_oof = np.load(str(chemprop_oof_path))
    chemprop_te  = np.load(str(chemprop_te_path))

    rae_stack = rae_fn(y_tr, oof_meta)
    rae_chemprop = rae_fn(y_tr, chemprop_oof)

    inv_stack    = 1.0 / (rae_stack + 1e-9)
    inv_chemprop = 1.0 / (rae_chemprop + 1e-9)
    w_stack    = inv_stack    / (inv_stack + inv_chemprop)
    w_chemprop = inv_chemprop / (inv_stack + inv_chemprop)

    blend_oof = w_stack * oof_meta + w_chemprop * chemprop_oof
    blend_te  = w_stack * te_preds + w_chemprop * chemprop_te
    blend_te  = np.clip(blend_te, y_tr.min()-0.5, y_tr.max()+0.5)

    blend_rae = rae_fn(y_tr, blend_oof)
    print(f'Stack RAE: {rae_stack:.4f}  Chemprop RAE: {rae_chemprop:.4f}')
    print(f'Blend weights: stack={w_stack:.3f}  chemprop={w_chemprop:.3f}')
    print(f'Blended OOF RAE: {blend_rae:.4f}')

    sub_blend = pd.DataFrame({'Molecule Name': te['name'].values, 'SMILES': te['smiles'].values, 'pEC50': blend_te})
    assert len(sub_blend) == 513 and sub_blend['pEC50'].notna().all()
    out_blend = SUBMISSIONS / '36b_grand_v6_chemprop_blend.csv'
    sub_blend.to_csv(out_blend, index=False)
    print(f'Saved blended: {out_blend}')
else:
    print('Chemprop OOF not found — skipping blend. Run nb08 first.')
    blend_rae = None
    blend_te  = None

Chemprop OOF not found — skipping blend. Run nb08 first.


In [7]:
# ── 7. Save stack-only submission ─────────────────────────────────────────────
sub = pd.DataFrame({'Molecule Name': te['name'].values, 'SMILES': te['smiles'].values, 'pEC50': te_preds})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out = SUBMISSIONS / '36_grand_v6.csv'
sub.to_csv(out, index=False)
print(f'\nSaved: {out}')
print(f'\n== Grand Ensemble v6 Summary ==')
print(f'  Models included:     {len(loaded)}')
print(f'  Nested CV RAE:       {nested_rae:.4f}')
print(f'  Grand v5 RAE:        0.5356')
print(f'  Delta vs v5:         {nested_rae - 0.5356:+.4f}')
if blend_rae:
    print(f'  Blended (w/Chemprop): {blend_rae:.4f}')
print(f'\nTest preds: mean={te_preds.mean():.3f}  std={te_preds.std():.3f}')


Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\36_grand_v6.csv

== Grand Ensemble v6 Summary ==
  Models included:     17
  Nested CV RAE:       0.2189
  Grand v5 RAE:        0.5356
  Delta vs v5:         -0.3167

Test preds: mean=4.619  std=0.118
